# Topology-aware, distribution-aware portfolio optimization

## Mathematical objective

At each rebalance date $t$, the project chooses portfolio weights

$$
\mathbf w_t=(w_{1,t},\ldots,w_{n,t})^\top\in\mathbb R^n
$$

using only information available before $t$. The final goal is not merely to
fit historical returns, but to produce stable out-of-sample allocations:

$$
\boxed{
\text{market data}
\longrightarrow
\text{structural information}
\longrightarrow
\text{asset selection}
\longrightarrow
\mathbf w_t
\longrightarrow
\text{future portfolio performance}
}
$$

This notebook combines two complementary geometric descriptions of the market:

$$
\begin{aligned}
\text{time geometry}
&:\quad
\text{persistent homology}
\longrightarrow \text{regime score }R_t,\\[2mm]
\text{asset geometry}
&:\quad
\text{Wasserstein distance and MMD}
\longrightarrow \text{asset clusters}.
\end{aligned}
$$

The complete pipeline is

$$
\text{prices}
\rightarrow \text{log returns}
\rightarrow
\begin{cases}
\text{topological regime estimation},\\
\text{distributional asset distances}
\end{cases}
\rightarrow
\text{representative assets}
\rightarrow
\text{spanning diagnostic}
\rightarrow
\text{regime-aware optimization}
\rightarrow
\text{walk-forward evaluation}.
$$

All estimates are causal: weights used after date $t$ are constructed only
from observations dated at or before $t$. Transaction costs are deducted.

> This notebook is a research prototype, not financial advice.


## Why Python is the recommended language

Python is the best primary language for this project because the full
mathematical pipeline can be expressed and tested in one environment:

- `pandas` and `NumPy` for prices, returns, vectors, and matrices;
- `SciPy` for Wasserstein distance and constrained optimization;
- `ripser` for Vietoris--Rips persistent homology;
- scikit-learn tools for preprocessing and statistical utilities;
- Jupyter for combining derivations, code, and experimental results.

A sensible future architecture is:

- **Python:** research, topology, clustering, optimization, and backtesting;
- **SQL:** persistent market and experiment data when the project grows;
- **C++ or Rust:** only for a measured production bottleneck.

The mathematical correctness of the model depends primarily on causal
estimation and valid out-of-sample testing, not on using a lower-level language.


## 0. Install the nonstandard packages

Run the installation cell once in each new environment. If Jupyter or Colab asks
for a kernel restart, restart it before continuing.


In [ ]:
%pip install -q yfinance ripser

## 1. Imports and reproducibility

Fixing a random seed makes stochastic steps reproducible. If two runs use the
same data, parameters, and seed, they should produce the same sampled bootstrap
windows and clustering initializations.


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf

from IPython.display import display
from ripser import ripser
from scipy.optimize import minimize
from scipy.spatial.distance import pdist
from scipy.stats import wasserstein_distance

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid", context="notebook")
np.random.seed(42)

## 2. Experiment configuration and information timing

Begin with `FAST_MODE=True`. After the entire notebook runs successfully, set it
to `False` to use more assets, more bootstrap samples, and a longer experiment.

Let $r_{i,s}$ denote the log return of asset $i$ on day $s$. At rebalance
index $t$, every estimator uses only the trailing information set

$$
\mathcal F_t=\sigma(r_{i,s}\colon s<t,\ i=1,\ldots,n).
$$

In the code this is enforced by slices of the form

```python
returns.iloc[t - lookback:t]
```

Python excludes the right endpoint, so observation $t$ is not used to create
weights applied beginning at $t$. This prevents look-ahead bias.


In [ ]:
@dataclass(frozen=True)
class Config:
    tickers: tuple[str, ...] = (
        "SPY", "QQQ", "IWM", "EFA", "EEM",
        "TLT", "IEF", "GLD", "VNQ",
        "XLE", "XLF", "XLV",
    )
    market_state_tickers: tuple[str, ...] = ("SPY", "QQQ", "IWM", "EFA")
    baseline_assets: tuple[str, ...] = ("SPY", "TLT", "GLD")
    benchmark: str = "SPY"
    start: str = "2016-01-01"
    end: str | None = None
    # Optional CSV fallback: first column must be Date; remaining columns are tickers.
    local_price_csv: str | None = None

    annualization: int = 252
    train_window: int = 504
    rebalance_every: int = 21
    tda_window: int = 60
    n_clusters: int = 4
    representatives_per_cluster: int = 1

    wasserstein_weight: float = 0.65
    mmd_weight: float = 0.35
    mmd_max_samples: int = 160

    # "robust" is quick. "bootstrap" follows a null-resampling threshold.
    tda_threshold_mode: str = "robust"
    tda_noise_mad_multiplier: float = 2.5
    tda_bootstrap_samples: int = 20
    tda_bootstrap_quantile: float = 0.95
    tda_block_length: int = 5

    base_risk_aversion: float = 8.0
    regime_risk_multiplier: float = 0.75
    max_weight_normal: float = 0.40
    max_weight_stress: float = 0.25
    turnover_penalty: float = 0.002
    transaction_cost_bps: float = 10.0
    covariance_ridge: float = 1e-6

    output_dir: str = "portfolio_outputs"
    fast_mode: bool = True

CFG = Config()
CFG

## 3. Prices and log returns

For adjusted closing price $P_{i,t}>0$, the one-period log return is

$$
r_{i,t}
=
\log\!\left(\frac{P_{i,t}}{P_{i,t-1}}\right)
=
\log P_{i,t}-\log P_{i,t-1}.
$$

Log returns are used because consecutive returns add across time:

$$
\sum_{s=t_0+1}^{t_1} r_{i,s}
=
\log\!\left(\frac{P_{i,t_1}}{P_{i,t_0}}\right).
$$

The code first attempts to download adjusted prices. A local adjusted-price CSV
can be supplied when online data are unavailable.


In [ ]:
def download_adjusted_close(
    tickers: tuple[str, ...],
    start: str,
    end: str | None,
    local_price_csv: str | None = None,
) -> pd.DataFrame:
    if local_price_csv:
        prices = pd.read_csv(local_price_csv, index_col=0, parse_dates=True)
        prices = prices.loc[:, [x for x in tickers if x in prices.columns]]
    else:
        raw = yf.download(
            list(tickers),
            start=start,
            end=end,
            auto_adjust=True,
            progress=False,
            threads=True,
            group_by="column",
        )
        if raw.empty:
            raise RuntimeError(
                "No data downloaded. Check internet access/tickers, or set "
                "CFG.local_price_csv to a local adjusted-price CSV."
            )

        if isinstance(raw.columns, pd.MultiIndex):
            if "Close" in raw.columns.get_level_values(0):
                prices = raw["Close"].copy()
            elif "Close" in raw.columns.get_level_values(1):
                prices = raw.xs("Close", axis=1, level=1).copy()
            else:
                raise KeyError("Could not find adjusted Close prices in yfinance output.")
        else:
            prices = raw[["Close"]].rename(columns={"Close": tickers[0]})

    prices = prices.sort_index().replace([np.inf, -np.inf], np.nan)
    missing_share = prices.isna().mean()
    keep = missing_share[missing_share <= 0.05].index
    dropped = sorted(set(tickers) - set(keep))
    if dropped:
        print("Dropped for >5% missing observations:", dropped)

    prices = prices.loc[:, keep].ffill(limit=3).dropna()
    return prices


prices = download_adjusted_close(
    CFG.tickers, CFG.start, CFG.end, CFG.local_price_csv
)
returns = np.log(prices).diff().dropna()

required = set(CFG.market_state_tickers) | set(CFG.baseline_assets) | {CFG.benchmark}
missing_required = sorted(required - set(returns.columns))
if missing_required:
    raise ValueError(f"Required assets missing after cleaning: {missing_required}")

print(f"Prices: {prices.index.min().date()} to {prices.index.max().date()}")
print(f"Observations: {len(returns):,}; assets: {returns.shape[1]}")
display(prices.tail())

In [ ]:
normalized_prices = prices / prices.iloc[0]
ax = normalized_prices.plot(figsize=(13, 6), lw=1.2, alpha=0.85)
ax.set(title="Growth of $1 before portfolio construction", ylabel="Normalized value")
plt.show()

## 4. Persistent-homology branch: market geometry through time

### 4.1 Rolling market-state point cloud

Choose $m$ broad market indices. Their returns on day $s$ form one market
state

$$
\mathbf x_s
=
\begin{bmatrix}
r_{1,s}\\
r_{2,s}\\
\vdots\\
r_{m,s}
\end{bmatrix}
\in\mathbb R^m.
$$

For a trailing window of $L$ days ending before rebalance $t$, define

$$
X_t
=
\left\{
\mathbf x_{t-L},\ldots,\mathbf x_{t-1}
\right\}.
$$

Thus, the **vertices are trading days**, while the coordinates describe the
simultaneous returns of the chosen indices.

### 4.2 Robust coordinate scaling

For coordinate $q$, use its median and median absolute deviation:

$$
\mathrm{MAD}_q
=
\mathrm{median}_{s}
\left|
x_{s,q}-\mathrm{median}_{u}(x_{u,q})
\right|.
$$

Let $s_q^{\mathrm{std}}$ be the sample standard deviation. The implemented
scale is

$$
a_q
=
\begin{cases}
1.4826\,\mathrm{MAD}_q,
&1.4826\,\mathrm{MAD}_q>\varepsilon_{\mathrm{num}},\\
s_q^{\mathrm{std}},
&s_q^{\mathrm{std}}>\varepsilon_{\mathrm{num}},\\
1,
&\text{otherwise}.
\end{cases}
$$

The robustly standardized coordinate is

$$
\widetilde x_{s,q}
=
\frac{x_{s,q}-\mathrm{median}_{u}(x_{u,q})}
{a_q}.
$$

The factor $1.4826$ makes MAD comparable to standard deviation for Gaussian
data.

### 4.3 Vietoris--Rips filtration and first homology

For distance threshold $\epsilon$, the Vietoris--Rips complex is

$$
\mathrm{VR}_{\epsilon}(X_t)
=
\left\{
\sigma\subseteq X_t:
d(\mathbf x,\mathbf y)\le \epsilon
\text{ for every }\mathbf x,\mathbf y\in\sigma
\right\}.
$$

Increasing $\epsilon$ produces a filtration

$$
\mathrm{VR}_{\epsilon_1}(X_t)
\subseteq
\mathrm{VR}_{\epsilon_2}(X_t)
\subseteq\cdots,
\qquad
\epsilon_1\le\epsilon_2\le\cdots.
$$

Using coefficients in $\mathbb F_2$, the first homology group is

$$
H_1
=
\frac{\ker(\partial_1)}{\mathrm{im}(\partial_2)}.
$$

Here, $\ker(\partial_1)$ contains closed edge cycles, while
$\mathrm{im}(\partial_2)$ contains cycles that are merely boundaries of
filled triangles. Their quotient identifies genuine one-dimensional holes.

The persistence diagram is

$$
D_t^{(1)}
=
\left\{(b_{j,t},d_{j,t})\right\}_{j=1}^{N_t},
$$

where loop $j$ appears at $b_{j,t}$, disappears at $d_{j,t}$, and has
lifetime

$$
\ell_{j,t}=d_{j,t}-b_{j,t}.
$$

### 4.4 Topological noise filter and regime score

Short-lived features are treated as topological noise:

$$
D_{t,\mathrm{signal}}^{(1)}
=
\left\{
(b_{j,t},d_{j,t})\in D_t^{(1)}:
\ell_{j,t}>\tau_t
\right\}.
$$

In robust mode, the implemented threshold is

$$
\tau_t
=
\mathrm{median}(\ell_t)
+1.4826\,c\,\mathrm{MAD}(\ell_t),
$$

where $c$ is `tda_noise_mad_multiplier`. In bootstrap mode, the threshold is
the configured quantile of the largest finite lifetime across moving-block
resamples. Importantly,
this step filters **features in the persistence diagram**; it does not delete
daily returns.

The surviving lifetimes are summarized by

$$
P_{2,t}
=
\left(
\sum_{j:\ell_{j,t}>\tau_t}\ell_{j,t}^{\,2}
\right)^{1/2}.
$$

The regime score compares the current summary with earlier summaries only.
First define the historical center

$$
m_t
=
\mathrm{median}\!\left(P_{2,1},\ldots,P_{2,t-1}\right)
$$

and historical MAD

$$
a_t
=
\mathrm{median}_{1\le u\le t-1}
\left|P_{2,u}-m_t\right|.
$$

Then

$$
R_t
=
\frac{P_{2,t}-m_t}{1.4826\,a_t}.
$$

Neither $m_t$ nor $a_t$ uses the current value $P_{2,t}$. The code
returns $0$ until enough history exists and falls back to sample standard
deviation if $a_t$ vanishes. A larger $R_t$ means that the recent
market-state cloud contains stronger or more unusual persistent loop structure.
This is a regime indicator, not automatically a crash prediction.


In [ ]:
def robust_scale_frame(frame: pd.DataFrame, eps: float = 1e-12) -> np.ndarray:
    x = frame.to_numpy(dtype=float)
    center = np.median(x, axis=0)
    mad = np.median(np.abs(x - center), axis=0)
    scale = 1.4826 * mad
    fallback = np.std(x, axis=0, ddof=1)
    scale = np.where(scale > eps, scale, np.where(fallback > eps, fallback, 1.0))
    return (x - center) / scale


def h1_lifetimes(point_cloud: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    diagram = ripser(point_cloud, maxdim=1)["dgms"][1]
    if diagram.size == 0:
        return diagram.reshape(0, 2), np.array([], dtype=float)
    diagram = diagram[np.isfinite(diagram[:, 1])]
    lifetimes = diagram[:, 1] - diagram[:, 0]
    return diagram, lifetimes


def moving_block_resample(
    x: np.ndarray, block_length: int, rng: np.random.Generator
) -> np.ndarray:
    n = len(x)
    starts = rng.integers(0, max(n - block_length + 1, 1), size=int(np.ceil(n / block_length)))
    blocks = [x[s : min(s + block_length, n)] for s in starts]
    return np.vstack(blocks)[:n]


def persistence_threshold(
    point_cloud: np.ndarray,
    lifetimes: np.ndarray,
    mode: str,
    rng: np.random.Generator,
) -> float:
    if lifetimes.size == 0:
        return np.inf

    if mode == "robust":
        med = np.median(lifetimes)
        mad = np.median(np.abs(lifetimes - med))
        return float(med + CFG.tda_noise_mad_multiplier * 1.4826 * mad)

    if mode == "bootstrap":
        null_maxima = []
        for _ in range(CFG.tda_bootstrap_samples):
            xb = moving_block_resample(point_cloud, CFG.tda_block_length, rng)
            _, lb = h1_lifetimes(xb)
            null_maxima.append(lb.max(initial=0.0))
        return float(np.quantile(null_maxima, CFG.tda_bootstrap_quantile))

    raise ValueError("mode must be 'robust' or 'bootstrap'")


def topology_summary(
    market_window: pd.DataFrame,
    mode: str = "robust",
    seed: int = 42,
) -> dict:
    x = robust_scale_frame(market_window)
    diagram, lifetimes = h1_lifetimes(x)
    threshold = persistence_threshold(
        x, lifetimes, mode=mode, rng=np.random.default_rng(seed)
    )
    keep = lifetimes > threshold
    surviving = lifetimes[keep]
    return {
        "diagram": diagram,
        "lifetimes": lifetimes,
        "threshold": threshold,
        "surviving_lifetimes": surviving,
        "n_surviving": int(keep.sum()),
        "total_persistence_l1": float(surviving.sum()),
        "total_persistence_l2": float(np.sqrt(np.square(surviving).sum())),
    }


def expanding_robust_z(values: list[float], min_history: int = 6) -> float:
    if len(values) <= min_history:
        return 0.0
    history = np.asarray(values[:-1], dtype=float)
    current = float(values[-1])
    center = np.median(history)
    mad = np.median(np.abs(history - center))
    scale = 1.4826 * mad
    if scale < 1e-12:
        scale = np.std(history, ddof=1)
    return 0.0 if scale < 1e-12 else float((current - center) / scale)


latest_market_window = returns.loc[:, CFG.market_state_tickers].iloc[-CFG.tda_window :]
latest_topology = topology_summary(
    latest_market_window, mode=CFG.tda_threshold_mode
)
{k: v for k, v in latest_topology.items() if k not in {"diagram", "lifetimes", "surviving_lifetimes"}}

In [ ]:
def plot_persistence_diagram(summary: dict, title: str) -> None:
    diagram = summary["diagram"]
    threshold = summary["threshold"]
    fig, ax = plt.subplots(figsize=(7, 6))
    if len(diagram):
        lifetimes = diagram[:, 1] - diagram[:, 0]
        signal = lifetimes > threshold
        ax.scatter(
            diagram[~signal, 0], diagram[~signal, 1],
            c="0.7", s=35, label="short-lived / filtered"
        )
        ax.scatter(
            diagram[signal, 0], diagram[signal, 1],
            c="crimson", s=55, label="persistent signal"
        )
        upper = float(diagram.max()) * 1.05
    else:
        upper = 1.0
    ax.plot([0, upper], [0, upper], "k--", lw=1, label="birth = death")
    ax.set(
        xlim=(0, upper), ylim=(0, upper),
        xlabel="Birth $b$", ylabel="Death $d$", title=title
    )
    ax.legend()
    plt.show()


plot_persistence_diagram(
    latest_topology,
    f"Latest $H_1$ persistence diagram ({latest_market_window.index[-1].date()})",
)

## 5. Distributional asset geometry: Wasserstein distance and MMD

Within a training window, asset $i$ is represented by its empirical return
distribution

$$
\widehat P_{i,t}
=
\frac{1}{L}\sum_{s=t-L}^{t-1}\delta_{r_{i,s}},
$$

where $\delta_x$ is a point mass at $x$. This retains more information than
representing the asset only by its sample mean and variance.

### 5.1 First Wasserstein distance

For one-dimensional distributions $P$ and $Q$,

$$
W_1(P,Q)
=
\int_0^1
\left|
F_P^{-1}(u)-F_Q^{-1}(u)
\right|\,du.
$$

It measures the minimum transportation cost required to deform one return
distribution into the other. The pairwise matrix is

$$
\left[D_t^{(W)}\right]_{ij}
=
W_1(\widehat P_{i,t},\widehat P_{j,t}).
$$

### 5.2 Maximum mean discrepancy

Let $k$ be a positive-definite kernel with feature map $\phi$ into a
reproducing-kernel Hilbert space $\mathcal H$. Then

$$
\mathrm{MMD}_k(P,Q)
=
\left\|
\mathbb E_{X\sim P}[\phi(X)]
-
\mathbb E_{Y\sim Q}[\phi(Y)]
\right\|_{\mathcal H}.
$$

Its squared kernel form is

$$
\mathrm{MMD}_k^2(P,Q)
=
\mathbb E[k(X,X')]
+\mathbb E[k(Y,Y')]
-2\mathbb E[k(X,Y)].
$$

The code uses a Gaussian RBF kernel with pair-specific median-distance bandwidth
and the biased empirical estimator (including kernel-matrix diagonal terms).
It passes $\sqrt{\max(\widehat{\mathrm{MMD}}^2,0)}$ to clustering.

The corresponding asset-distance matrix is

$$
\left[D_t^{(M)}\right]_{ij}
=
\mathrm{MMD}_k(\widehat P_{i,t},\widehat P_{j,t}).
$$

After robust normalization, the two geometries are combined:

$$
D_t
=
\omega_W\widetilde D_t^{(W)}
+\omega_M\widetilde D_t^{(M)},
\qquad
\omega_W,\omega_M\ge 0,
\quad
\omega_W+\omega_M=1.
$$

K-medoids is appropriate because $D_t$ is a precomputed, potentially
non-Euclidean distance matrix. Unlike a K-means centroid, each medoid is an
actual traded asset.


In [ ]:
def rbf_mmd_distance(
    x: np.ndarray,
    y: np.ndarray,
    max_samples: int,
    rng: np.random.Generator,
) -> float:
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    if len(x) > max_samples:
        x = rng.choice(x, size=max_samples, replace=False)
    if len(y) > max_samples:
        y = rng.choice(y, size=max_samples, replace=False)

    pooled = np.concatenate([x, y])[:, None]
    pairwise = pdist(pooled, metric="euclidean")
    positive = pairwise[pairwise > 0]
    bandwidth = np.median(positive) if positive.size else 1.0
    gamma = 1.0 / (2.0 * max(bandwidth, 1e-12) ** 2)

    kxx = np.exp(-gamma * (x[:, None] - x[None, :]) ** 2).mean()
    kyy = np.exp(-gamma * (y[:, None] - y[None, :]) ** 2).mean()
    kxy = np.exp(-gamma * (x[:, None] - y[None, :]) ** 2).mean()
    return float(np.sqrt(max(kxx + kyy - 2.0 * kxy, 0.0)))


def pairwise_distribution_distances(
    train_returns: pd.DataFrame, seed: int = 42
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    names = list(train_returns.columns)
    n = len(names)
    dw = np.zeros((n, n), dtype=float)
    dm = np.zeros((n, n), dtype=float)
    rng = np.random.default_rng(seed)

    for i in range(n):
        xi = train_returns.iloc[:, i].dropna().to_numpy()
        for j in range(i + 1, n):
            xj = train_returns.iloc[:, j].dropna().to_numpy()
            dw[i, j] = dw[j, i] = wasserstein_distance(xi, xj)
            dm[i, j] = dm[j, i] = rbf_mmd_distance(
                xi, xj, CFG.mmd_max_samples, rng
            )

    def normalize_distance(d: np.ndarray) -> np.ndarray:
        upper = d[np.triu_indices_from(d, k=1)]
        positive = upper[upper > 0]
        scale = np.median(positive) if positive.size else 1.0
        return d / max(scale, 1e-12)

    combined = (
        CFG.wasserstein_weight * normalize_distance(dw)
        + CFG.mmd_weight * normalize_distance(dm)
    )
    return (
        pd.DataFrame(dw, index=names, columns=names),
        pd.DataFrame(dm, index=names, columns=names),
        pd.DataFrame(combined, index=names, columns=names),
    )


def k_medoids(
    distance: pd.DataFrame, n_clusters: int, max_iter: int = 100
) -> tuple[pd.Series, list[str]]:
    names = list(distance.index)
    d = distance.to_numpy()
    n = len(names)
    k = min(max(1, n_clusters), n)

    medoids = [int(np.argmin(d.sum(axis=1)))]
    while len(medoids) < k:
        nearest = d[:, medoids].min(axis=1)
        nearest[medoids] = -np.inf
        medoids.append(int(np.argmax(nearest)))

    for _ in range(max_iter):
        labels = np.argmin(d[:, medoids], axis=1)
        new_medoids = []
        for cluster in range(k):
            members = np.where(labels == cluster)[0]
            if len(members) == 0:
                candidates = [i for i in range(n) if i not in new_medoids]
                new_medoids.append(max(candidates, key=lambda i: d[i, medoids].min()))
            else:
                within = d[np.ix_(members, members)]
                new_medoids.append(int(members[np.argmin(within.sum(axis=1))]))
        if new_medoids == medoids:
            break
        medoids = new_medoids

    labels = np.argmin(d[:, medoids], axis=1)
    return pd.Series(labels, index=names, name="cluster"), [names[i] for i in medoids]


def select_representatives(
    train_returns: pd.DataFrame,
    distance: pd.DataFrame,
    labels: pd.Series,
    per_cluster: int,
) -> list[str]:
    annual_mean = train_returns.mean() * CFG.annualization
    annual_vol = train_returns.std(ddof=1) * np.sqrt(CFG.annualization)
    sharpe = annual_mean / annual_vol.replace(0, np.nan)
    selected: list[str] = []

    for cluster in sorted(labels.unique()):
        members = labels.index[labels == cluster].tolist()
        centrality = distance.loc[members, members].mean(axis=1)
        centrality_z = (centrality - centrality.mean()) / (centrality.std(ddof=0) + 1e-12)
        sharpe_z = (sharpe.loc[members] - sharpe.loc[members].mean()) / (
            sharpe.loc[members].std(ddof=0) + 1e-12
        )
        # Prefer a central distribution, with a modest quality tilt.
        score = -centrality_z + 0.25 * sharpe_z.fillna(0.0)
        selected.extend(score.nlargest(min(per_cluster, len(members))).index.tolist())

    return sorted(set(selected))

### 5.3 K-medoids clustering and representative assets

Let $c(i)\in\{1,\ldots,K\}$ be asset $i$'s cluster and let $m_k$ be the
medoid index of cluster $k$. K-medoids approximately solves

$$
\min_{\{c(i)\},\,\{m_k\}}
\sum_{i=1}^{n} D_{i,m_{c(i)}}.
$$

The medoid

$$
m_k
\in
\arg\min_{j\in C_k}
\sum_{i\in C_k}D_{ij}
$$

is the asset most centrally located within cluster $C_k$. Selecting one
medoid per cluster reduces redundancy while retaining different regions of the
asset-distribution geometry.


In [ ]:
current_train = returns.iloc[-CFG.train_window :]
current_w, current_mmd, current_distance = pairwise_distribution_distances(current_train)
current_labels, current_medoids = k_medoids(current_distance, CFG.n_clusters)
current_selected = select_representatives(
    current_train,
    current_distance,
    current_labels,
    CFG.representatives_per_cluster,
)

cluster_table = pd.DataFrame({
    "cluster": current_labels,
    "is_medoid": current_labels.index.isin(current_medoids),
    "selected": current_labels.index.isin(current_selected),
}).sort_values(["cluster", "selected", "is_medoid"], ascending=[True, False, False])
display(cluster_table)

plt.figure(figsize=(10, 8))
sns.heatmap(current_distance, cmap="viridis", square=True)
plt.title("Combined normalized Wasserstein/MMD distance")
plt.show()

## 6. Mean--variance spanning diagnostic

For a candidate asset set with estimated mean vector
$\widehat{\boldsymbol\mu}\in\mathbb R^n$ and covariance matrix
$\widehat\Sigma\in\mathbb R^{n\times n}$, the long-only minimum-variance
portfolio at target return $r_\star$ solves

$$
\begin{aligned}
\min_{\mathbf w\in\mathbb R^n}
\quad&
\mathbf w^\top\widehat\Sigma\mathbf w\\
\text{subject to}\quad&
\mathbf w^\top\widehat{\boldsymbol\mu}\ge r_\star,\\
&
\mathbf 1^\top\mathbf w=1,\\
&
\mathbf w\ge\mathbf 0.
\end{aligned}
$$

Repeating this problem over a grid of $r_\star$ values traces an estimated
efficient frontier.

Let $\sigma_{\mathrm{base}}(r_\star)$ and
$\sigma_{\mathrm{selected}}(r_\star)$ be the minimum volatilities obtained
from a baseline universe and the selected universe. A useful descriptive
improvement is

$$
\Delta\sigma(r_\star)
=
\sigma_{\mathrm{base}}(r_\star)
-
\sigma_{\mathrm{selected}}(r_\star).
$$

Positive $\Delta\sigma$ means the selected set attains the target return with
less estimated volatility. This notebook computes an **economic frontier
diagnostic**, not the formal Huberman--Kandel statistical spanning test.
Out-of-sample performance remains the more important evidence.


In [ ]:
def annual_moments(frame: pd.DataFrame) -> tuple[pd.Series, pd.DataFrame]:
    mu = frame.mean() * CFG.annualization
    cov = frame.cov() * CFG.annualization
    cov = cov + np.eye(len(cov)) * CFG.covariance_ridge
    return mu, cov


def minimum_volatility_for_target(
    mu: pd.Series, cov: pd.DataFrame, target_return: float
) -> float:
    n = len(mu)

    def objective(w):
        return float(w @ cov.to_numpy() @ w)

    constraints = [
        {"type": "eq", "fun": lambda w: np.sum(w) - 1.0},
        {"type": "ineq", "fun": lambda w: float(w @ mu.to_numpy() - target_return)},
    ]
    result = minimize(
        objective,
        x0=np.repeat(1.0 / n, n),
        method="SLSQP",
        bounds=[(0.0, 1.0)] * n,
        constraints=constraints,
        options={"maxiter": 1000, "ftol": 1e-12},
    )
    return np.sqrt(max(result.fun, 0.0)) if result.success else np.nan


def efficient_frontier(
    train_returns: pd.DataFrame, targets: np.ndarray
) -> pd.Series:
    mu, cov = annual_moments(train_returns)
    values = [minimum_volatility_for_target(mu, cov, target) for target in targets]
    return pd.Series(values, index=targets)


baseline = [x for x in CFG.baseline_assets if x in current_train.columns]
expanded = sorted(set(baseline) | set(current_selected))
mu_base = current_train[baseline].mean() * CFG.annualization
mu_full = current_train[expanded].mean() * CFG.annualization
lower = max(float(mu_base.min()), float(mu_full.min()))
upper = min(float(mu_base.max()), float(mu_full.max()))
targets = np.linspace(lower, upper, 25)

frontier_base = efficient_frontier(current_train[baseline], targets)
frontier_full = efficient_frontier(current_train[expanded], targets)
spanning_table = pd.DataFrame({
    "baseline_vol": frontier_base,
    "expanded_vol": frontier_full,
    "vol_reduction": frontier_base - frontier_full,
})
display(spanning_table.describe().loc[["mean", "min", "max"]])

plt.figure(figsize=(8, 6))
plt.plot(frontier_base, targets, label=f"Baseline: {baseline}", lw=2)
plt.plot(frontier_full, targets, label=f"Expanded: {expanded}", lw=2)
plt.xlabel("Annualized volatility")
plt.ylabel("Annualized expected return")
plt.title("Current-window long-only efficient frontiers")
plt.legend()
plt.show()

## 7. Regime-aware portfolio optimizer

For the assets selected at rebalance $t$, the code annualizes the trailing
sample mean:

$$
\widehat{\boldsymbol\mu}_t
=
A\frac{1}{L}\sum_{s=t-L}^{t-1}\mathbf r_s,
$$

where $A=252$. It annualizes the sample covariance and shrinks it toward its
diagonal to obtain $\widehat\Sigma_t^{\mathrm{shrunk}}$.

The new portfolio solves

$$
\begin{aligned}
\mathbf w_t^\star
\in
\arg\min_{\mathbf w}
\quad&
-\widehat{\boldsymbol\mu}_t^\top\mathbf w
+\gamma_t\,\mathbf w^\top\widehat\Sigma_t^{\mathrm{shrunk}}\mathbf w
+\eta\left\|\mathbf w-\mathbf w_{t-1}\right\|_1\\
\text{subject to}\quad&
\mathbf 1^\top\mathbf w=1,\\
&
0\le w_i\le u_t.
\end{aligned}
$$

The three objective terms represent:

$$
\underbrace{-\widehat{\boldsymbol\mu}_t^\top\mathbf w}_{\text{reward expected return}}
\;+\;
\underbrace{\gamma_t\mathbf w^\top\widehat\Sigma_t^{\mathrm{shrunk}}\mathbf w}_{\text{penalize risk}}
\;+\;
\underbrace{\eta\|\mathbf w-\mathbf w_{t-1}\|_1}_{\text{penalize turnover}}.
$$

Risk aversion responds to the topological regime score:

$$
\gamma_t
=
\gamma_0
\left(1+\beta\max\{R_t,0\}\right).
$$

Here $\beta$ is `regime_risk_multiplier`. The maximum weight is $0.40$ in
the normal state and $0.25$ when $R_t\ge1$, but never below $1/n_t$, which
keeps the constraints feasible. Therefore, larger $R_t$ produces a more
conservative allocation.


In [ ]:
def shrink_covariance(sample_cov: np.ndarray, strength: float) -> np.ndarray:
    diagonal_target = np.diag(np.diag(sample_cov))
    shrunk = (1.0 - strength) * sample_cov + strength * diagonal_target
    return shrunk + np.eye(len(sample_cov)) * CFG.covariance_ridge


def optimize_weights(
    train_returns: pd.DataFrame,
    previous_weights: pd.Series | None,
    regime_z: float,
) -> pd.Series:
    assets = list(train_returns.columns)
    n = len(assets)
    mu = train_returns.mean().to_numpy() * CFG.annualization
    sample_cov = train_returns.cov().to_numpy() * CFG.annualization

    stress = max(float(regime_z), 0.0)
    shrinkage = float(np.clip(0.10 + 0.10 * stress, 0.10, 0.60))
    cov = shrink_covariance(sample_cov, shrinkage)
    gamma = CFG.base_risk_aversion * (1.0 + CFG.regime_risk_multiplier * stress)
    max_weight = CFG.max_weight_stress if stress >= 1.0 else CFG.max_weight_normal
    max_weight = max(max_weight, 1.0 / n)

    if previous_weights is None:
        previous = np.repeat(1.0 / n, n)
    else:
        previous = previous_weights.reindex(assets).fillna(0.0).to_numpy()
        if previous.sum() <= 1e-12:
            previous = np.repeat(1.0 / n, n)
        else:
            previous = previous / previous.sum()

    def objective(w):
        expected_return = float(w @ mu)
        variance = float(w @ cov @ w)
        turnover = float(np.abs(w - previous).sum())
        return -expected_return + gamma * variance + CFG.turnover_penalty * turnover

    result = minimize(
        objective,
        x0=np.clip(previous, 0.0, max_weight),
        method="SLSQP",
        bounds=[(0.0, max_weight)] * n,
        constraints=[{"type": "eq", "fun": lambda w: np.sum(w) - 1.0}],
        options={"maxiter": 1000, "ftol": 1e-12},
    )

    if not result.success:
        print("Optimizer fallback:", result.message)
        weights = np.repeat(1.0 / n, n)
    else:
        weights = np.clip(result.x, 0.0, None)
        weights = weights / weights.sum()
    return pd.Series(weights, index=assets, name="weight")


latest_weights = optimize_weights(
    current_train[current_selected],
    previous_weights=None,
    regime_z=0.0,
)
display(latest_weights.sort_values(ascending=False).to_frame())

## 8. Strict walk-forward backtest

At rebalance date $t_k$, the algorithm performs the following causal sequence:

1. Form the training sample
   $\mathcal D_{t_k}=\{\mathbf r_s:t_k-L\le s<t_k\}$.
2. Compute Wasserstein/MMD distances using only $\mathcal D_{t_k}$.
3. Cluster assets and select one representative per cluster.
4. Compute the trailing topological score $R_{t_k}$.
5. Estimate $\widehat{\boldsymbol\mu}_{t_k}$ and
   $\widehat\Sigma_{t_k}$, then solve for $\mathbf w_{t_k}$.
6. Hold those weights over the next out-of-sample interval.
7. Deduct transaction costs at the rebalance.

For day $s$ in the holding interval, the gross portfolio return is

$$
r_{p,s}^{\mathrm{gross}}
=
\mathbf w_{t_k}^\top\mathbf r_s.
$$

The implementation records the full reallocation distance

$$
\mathrm{Q}_{t_k}
=
\left\|
\mathbf w_{t_k}-\mathbf w_{t_{k-1}}
\right\|_1.
$$

If the proportional cost rate is $c$, the rebalance cost is

$$
C_{t_k}=c\,\mathrm{Q}_{t_k}.
$$

It is deducted from the first log return of the new holding period using
$r_{p,t_k}^{\mathrm{net}}\approx r_{p,t_k}^{\mathrm{gross}}-C_{t_k}$.
Conventional one-way turnover is $\tfrac12\mathrm{Q}_{t_k}$, so this project's
reported convention must remain explicit.

The notebook compares four strategies:

- **Full model:** topology, distributional clustering, and regime-aware weights;
- **All-asset MVO:** conventional optimization without topology or selection;
- **Equal weight:** $w_i=1/n$ for all available assets;
- **SPY:** a simple investable benchmark.


In [ ]:
def run_walk_forward(
    all_returns: pd.DataFrame,
    use_tda: bool,
    use_clustering: bool,
    equal_weight: bool = False,
    label: str = "strategy",
) -> dict:
    start_t = max(CFG.train_window, CFG.tda_window)
    if CFG.fast_mode:
        start_t = max(start_t, len(all_returns) - 5 * CFG.annualization)

    portfolio_chunks = []
    weight_rows = []
    topology_rows = []
    previous_full = pd.Series(0.0, index=all_returns.columns)
    topology_history: list[float] = []

    rebalance_points = list(range(start_t, len(all_returns), CFG.rebalance_every))
    for step_number, t in enumerate(rebalance_points):
        train = all_returns.iloc[t - CFG.train_window : t]
        hold = all_returns.iloc[t : min(t + CFG.rebalance_every, len(all_returns))]
        if hold.empty:
            continue

        if use_tda:
            market_window = all_returns.loc[
                :, CFG.market_state_tickers
            ].iloc[t - CFG.tda_window : t]
            summary = topology_summary(
                market_window,
                mode=CFG.tda_threshold_mode,
                seed=42 + step_number,
            )
            topology_history.append(summary["total_persistence_l2"])
            regime_z = expanding_robust_z(topology_history)
        else:
            summary = {
                "total_persistence_l2": 0.0,
                "n_surviving": 0,
                "threshold": np.nan,
            }
            regime_z = 0.0

        if use_clustering:
            _, _, distance = pairwise_distribution_distances(
                train, seed=42 + step_number
            )
            labels, _ = k_medoids(distance, CFG.n_clusters)
            selected = select_representatives(
                train,
                distance,
                labels,
                CFG.representatives_per_cluster,
            )
        else:
            selected = list(train.columns)

        if equal_weight:
            selected_weights = pd.Series(
                1.0 / len(selected), index=selected, name="weight"
            )
        else:
            selected_weights = optimize_weights(
                train[selected],
                previous_weights=previous_full,
                regime_z=regime_z,
            )

        full_weights = pd.Series(0.0, index=all_returns.columns)
        full_weights.loc[selected_weights.index] = selected_weights
        turnover = float(np.abs(full_weights - previous_full).sum())

        realized = hold @ full_weights
        realized = realized.copy()
        realized.iloc[0] -= turnover * CFG.transaction_cost_bps / 10_000.0
        portfolio_chunks.append(realized.rename(label))

        row = full_weights.to_dict()
        row.update({
            "date": hold.index[0],
            "turnover": turnover,
            "regime_z": regime_z,
            "n_selected": len(selected),
        })
        weight_rows.append(row)
        topology_rows.append({
            "date": hold.index[0],
            "total_persistence_l2": summary["total_persistence_l2"],
            "n_surviving": summary["n_surviving"],
            "threshold": summary["threshold"],
            "regime_z": regime_z,
        })
        previous_full = full_weights

    strategy_returns = pd.concat(portfolio_chunks).sort_index()
    weights = pd.DataFrame(weight_rows).set_index("date").sort_index()
    topology = pd.DataFrame(topology_rows).set_index("date").sort_index()
    return {"returns": strategy_returns, "weights": weights, "topology": topology}

### 8.1 Run all strategies

The following cell executes the same evaluation dates for every strategy so
that differences are not caused by mismatched samples.


In [ ]:
full_model = run_walk_forward(
    returns, use_tda=True, use_clustering=True, label="Full model"
)
conventional_mvo = run_walk_forward(
    returns, use_tda=False, use_clustering=False, label="All-asset MVO"
)
equal_weight = run_walk_forward(
    returns,
    use_tda=False,
    use_clustering=False,
    equal_weight=True,
    label="Equal weight",
)

common_start = max(
    full_model["returns"].index.min(),
    conventional_mvo["returns"].index.min(),
    equal_weight["returns"].index.min(),
)
strategy_returns = pd.concat(
    [
        full_model["returns"],
        conventional_mvo["returns"],
        equal_weight["returns"],
        returns.loc[common_start:, CFG.benchmark].rename(CFG.benchmark),
    ],
    axis=1,
).dropna()
strategy_returns.tail()

## 9. Performance, drawdown, turnover, and regime diagnostics

For daily **log** portfolio returns $r_{p,1},\ldots,r_{p,T}$, cumulative
wealth is

$$
V_t
=
\exp\left(\sum_{s=1}^{t}r_{p,s}\right),
\qquad V_0=1.
$$

Using $A=252$ trading days per year, annualized return and volatility are
estimated as

$$
\widehat\mu_{\mathrm{ann}}
=
A\,\overline r_p,
\qquad
\widehat\sigma_{\mathrm{ann}}
=
\sqrt{A}\,s(r_p).
$$

With daily risk-free rate $r_{f,s}$, the annualized Sharpe ratio is

$$
\widehat{\mathrm{SR}}
=
\sqrt{A}\,
\frac{\overline{(r_p-r_f)}}{s(r_p-r_f)}.
$$

Running peak wealth and drawdown are

$$
M_t=\max_{0\le u\le t}V_u,
\qquad
\mathrm{DD}_t=\frac{V_t}{M_t}-1.
$$

Maximum drawdown is

$$
\mathrm{MDD}
=
\min_{1\le t\le T}\mathrm{DD}_t.
$$

These metrics must be read together: a strategy with a high return but extreme
drawdown, unstable weights, or excessive turnover may not be practically
superior.


In [ ]:
def max_drawdown(series: pd.Series) -> float:
    wealth = np.exp(series.cumsum())
    running_peak = wealth.cummax().clip(lower=1.0)
    drawdown = wealth / running_peak - 1.0
    return float(drawdown.min())


def performance_table(
    log_return_frame: pd.DataFrame,
    turnover_by_strategy: dict[str, pd.Series] | None = None,
) -> pd.DataFrame:
    rows = {}
    for name, x in log_return_frame.items():
        years = len(x) / CFG.annualization
        total_growth = float(np.exp(x.sum()))
        cagr = total_growth ** (1.0 / years) - 1.0
        vol = float(x.std(ddof=1) * np.sqrt(CFG.annualization))
        annual_return = float(x.mean() * CFG.annualization)
        sharpe = annual_return / vol if vol > 0 else np.nan
        downside = x[x < 0].std(ddof=1) * np.sqrt(CFG.annualization)
        sortino = annual_return / downside if downside > 0 else np.nan
        rows[name] = {
            "CAGR": cagr,
            "Annual volatility": vol,
            "Sharpe (rf=0)": sharpe,
            "Sortino (rf=0)": sortino,
            "Max drawdown": max_drawdown(x),
            "Growth of $1": total_growth,
        }

    table = pd.DataFrame(rows).T
    if turnover_by_strategy:
        table["Mean rebalance turnover"] = pd.Series({
            k: float(v.mean()) for k, v in turnover_by_strategy.items()
        })
    return table


turnover_series = {
    "Full model": full_model["weights"]["turnover"],
    "All-asset MVO": conventional_mvo["weights"]["turnover"],
    "Equal weight": equal_weight["weights"]["turnover"],
}
metrics = performance_table(strategy_returns, turnover_series)
display(
    metrics.style.format({
        "CAGR": "{:.2%}",
        "Annual volatility": "{:.2%}",
        "Sharpe (rf=0)": "{:.2f}",
        "Sortino (rf=0)": "{:.2f}",
        "Max drawdown": "{:.2%}",
        "Growth of $1": "{:.2f}",
        "Mean rebalance turnover": "{:.2%}",
    })
)

In [ ]:
wealth = np.exp(strategy_returns.cumsum())
running_peaks = wealth.cummax().clip(lower=1.0)
drawdowns = wealth.div(running_peaks).sub(1.0)

fig, axes = plt.subplots(2, 1, figsize=(13, 10), sharex=True)
wealth.plot(ax=axes[0], lw=2)
axes[0].set(title="Strict walk-forward growth of $1", ylabel="Portfolio value")
drawdowns.plot(ax=axes[1], lw=1.5)
axes[1].set(title="Drawdowns", ylabel="Drawdown", xlabel="")
plt.tight_layout()
plt.show()

In [ ]:
topology_history = full_model["topology"]
fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)
topology_history["total_persistence_l2"].plot(
    ax=axes[0], color="darkorange", lw=1.8
)
axes[0].set(title="Causal topological persistence signal", ylabel="$P_2$")
topology_history["regime_z"].plot(ax=axes[1], color="crimson", lw=1.8)
axes[1].axhline(1.0, color="black", ls="--", lw=1)
axes[1].set(title="Expanding robust regime z-score", ylabel="$R_t$")
plt.tight_layout()
plt.show()

In [ ]:
asset_columns = [c for c in returns.columns if c in full_model["weights"].columns]
fig, ax = plt.subplots(figsize=(13, 7))
full_model["weights"][asset_columns].plot.area(
    ax=ax, stacked=True, alpha=0.85, linewidth=0
)
ax.set(title="Full-model weights at each rebalance", ylabel="Weight", ylim=(0, 1))
ax.legend(loc="center left", bbox_to_anchor=(1.01, 0.5))
plt.tight_layout()
plt.show()

## 10. Save reproducible outputs

The output folder records the exact numerical objects behind the figures:

- daily out-of-sample strategy returns;
- summary performance metrics;
- weights at every rebalance;
- turnover and transaction costs;
- topological regime history.

Saving these objects makes later ablation tests directly comparable.


In [ ]:
output_dir = Path(CFG.output_dir)
output_dir.mkdir(parents=True, exist_ok=True)

strategy_returns.to_csv(output_dir / "daily_log_returns.csv")
metrics.to_csv(output_dir / "performance_metrics.csv")
full_model["weights"].to_csv(output_dir / "full_model_weights.csv")
full_model["topology"].to_csv(output_dir / "topology_history.csv")
cluster_table.to_csv(output_dir / "latest_clusters.csv")

print(f"Saved outputs to: {output_dir.resolve()}")

## 11. Statistical interpretation and next experiments

The full method is useful only if it improves decisions on data that were not
used to estimate or tune it. One favorable backtest is not sufficient evidence.

The core research hypothesis is

$$
\begin{aligned}
H_0:\;&
\text{topology-aware filtering and distribution-aware selection}\\
&\text{do not improve out-of-sample portfolio performance},\\[1mm]
H_1:\;&
\text{they improve risk-adjusted performance or stability after costs}.
\end{aligned}
$$

“Improvement” should be specified before testing, for example:

$$
\Delta\mathrm{Sharpe}>0,\qquad
\Delta|\mathrm{MDD}|<0,\qquad
\Delta\mathrm{Turnover}\le 0,
$$

subject to acceptable realized return.

Recommended next experiments:

1. Perform ablations by changing only one component at a time: persistent
   homology, Wasserstein distance, MMD, or clustering.
2. Test multiple non-overlapping periods and asset universes.
3. Add a cash or risk-free asset and realistic fund-specific trading costs.
4. Compare robust and bootstrap persistence thresholds.
5. Use nested walk-forward tuning so hyperparameters are selected only inside
   each training period.
6. Report uncertainty using moving-block bootstrap confidence intervals.

The central question is

$$
\boxed{
\begin{gathered}
\text{Does persistent market topology plus distribution-aware asset selection}\\
\text{produce more stable out-of-sample portfolios than conventional methods?}
\end{gathered}
}
$$
